# 07 Linear Probe 평가

체크포인트에서 feature를 추출하고 linear classifier를 학습해 표현력을 측정합니다.
feature 추출은 모든 GPU에서 병렬로 진행됩니다.

In [ ]:
import torch
from src.experiments import load_cfg, deep_update

# ── GPU 자동 감지 ───────────────────────────────────────────────
n_gpu = torch.cuda.device_count()
print(f"감지된 GPU 수: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory // 1024**3} GB")
if n_gpu == 0:
    print("  (GPU 없음 — CPU로 실행됩니다)")


In [ ]:
from src.experiments import load_cfg, deep_update, launch_eval_linear

cfg = load_cfg("configs/cifar10.yaml")
checkpoint_path = "outputs/checkpoints/cifar10_full_sdd_last.pt"

launch_eval_linear(
    cfg,
    checkpoint_path=checkpoint_path,
    probe_epochs=50,
    probe_lr=1e-3,
    num_processes=None,
)


In [ ]:
# baseline vs SDD 비교
from src.experiments import load_cfg, deep_update, launch_eval_linear

cfg_b = load_cfg("configs/cifar10.yaml")
cfg_b = deep_update(cfg_b, {
    "sdd": {"enabled": False, "lambda_distill": 0.0,
            "use_centering": False, "use_sharpening": False,
            "use_gating": False, "use_projection_head": False},
})

print("\n▶ Baseline linear probe:")
launch_eval_linear(cfg_b, "outputs/checkpoints/cifar10_baseline_mse_only_last.pt",
                   num_processes=None)

print("\n▶ Full SDD linear probe:")
launch_eval_linear(load_cfg("configs/cifar10.yaml"),
                   "outputs/checkpoints/cifar10_full_sdd_last.pt",
                   num_processes=None)
